# The Structured Event Extractor

Today's drill is to deconstruct the mathematical engines driving modern search by implementing keyword matching \
algorithms from scratch and benchmarking them against dense semantic vectors.

### Core Math Specs

Task A: TF-IDF from Scratch

You cannot use `scikit-learn`. Implement the classic TF-IDF calculation using `numpy`.

* **Term Frequency ($TF$):** $$TF = \frac{\text{Count of term in doc}}{\text{Total terms in doc}}$$

* **Inverse Document Frequency ($IDF$):** $$IDF = \log\left(\frac{N}{DF}\right) + 1$$
  *(where $N$ is total documents, and $DF$ is the number of documents containing the term. Use natural log `np.log`)*.

Compute the TF-IDF matrix for the corpus, tokenize the query, vectorise the query using the corpus vocabulary, and use \
Cosine Similarity to rank the top 2 documents.

$$\text{Cosine Similarity}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

Task B: BM25 (Best Matching 25) from Scratch

BM25 improves on TF-IDF by adding term frequency saturation (preventing a word repeated 100 times from completely \
biasing the score) and document length normalization.

Formula for a single query term $q$ in document $D$:

$$\text{Score}(D, q) = IDF(q) \times \frac{TF(D, q) \times (k_1 + 1)}{TF(D, q) + k_1 \times \left(1 - b + b \times 
\frac{|D|}{\text{avgdl}}\right)}$$

### Parameters & Variables:
* **Parameters:** Set $k_1 = 1.5$, $b = 0.75$.

* $|D|$ is the length of the document in tokens.

* $\text{avgdl}$ is the average document length across the entire corpus.

* **IDF Variant:** $$IDF(q) = \log\left(\frac{N - DF(q) + 0.5}{DF(q) + 0.5} + 1\right)$$ *(Standard BM25 IDF variant)*.

Sum the scores for all words in the query that appear in the document. Rank the top 2 documents.

Task C: Dense Embedding Retrieval

Use your local Ollama engine (e.g., pulling an embedding model like `nomic-embed-text` or `all-minilm` using \
`ollama.embeddings()`) or `sentence-transformers` to get dense 1D arrays for the corpus and the query. 

Compute the Cosine Similarity matrix using NumPy and rank the top 2 documents.

### Tasks

Preprocess: Write a simple tokenization step (lowercase and split by whitespace, ignore punctuation for simplicity today).

Build the Matrix: Implement the pure NumPy loops or vectorizations to generate the TF-IDF and BM25 score arrays.

Embed: Get the dense vectors from your local model.

The Benchmark Printout: Your script must print out the top 2 ranked documents along with their scores for all three methods.

### Expected Verification Output Structure

=== TF-IDF Search Results ===
* **Rank 1:** Doc ID X (Score: `0.XXXX`) - *"Text..."*
* **Rank 2:** Doc ID Y (Score: `0.XXXX`) - *"Text..."*

=== BM25 Search Results ===
* **Rank 1:** Doc ID X (Score: `0.XXXX`) - *"Text..."*

=== Dense Embedding Search Results ===
* **Rank 1:** Doc ID X (Score: `0.XXXX`) - *"Text..."*

### Import Dependencies

In [74]:
import numpy as np
import requests

### Corpus and Query
Define the evaluation dataset and target query. The corpus is engineered to test the limits of exact keyword matching \
against semantic encoding.

In [75]:
corpus = [
    "The quick brown fox jumps over the lazy dog",
    "Python is an interpreted, high-level programming language for general-purpose programming.",
    "The search engine uses a vector database to retrieve relevant documents quickly.",
    "Information retrieval systems evaluate term frequency and inverse document frequency.",
    "Dogs and foxes are both animals, but cats are completely different mammals."
]

query = "retrieval systems for programming language"

### Preprocessing and Helpers
Implement primitive whitespace tokenization, construct the global vocabulary matrix bounds, and establish our spatial \
proximity metric (Cosine Similarity).

In [76]:
def tokenize(text):
    return text.lower().split()

In [77]:
tokenized_corpus = [tokenize(doc) for doc in corpus]
tokenized_query = tokenize(query)

In [78]:
# Build Vocabulary from corpus (ordered list for indexing)
vocabulary = sorted(list(set(word for doc in tokenized_corpus for word in doc)))
vocab_idx = {word: i for i, word in enumerate(vocabulary)}

N = len(corpus)
avgdl = np.mean([len(doc) for doc in tokenized_corpus])

In [79]:
def cosine_similarity(v1, v2):
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)

    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
    return np.dot(v1, v2) / (norm_v1 * norm_v2)

### Task A: TF-IDF Retrieval form Scratch
Compute the base lexical frequency matrix. This assigns absolute statistical weights to term occurrences across \
documents without scaling dampening parameters.

In [80]:
# Compute DF for Vocabulary
df_tfidf = {word: sum(1 for doc in tokenized_corpus if word in doc) for word in vocabulary}
idf_tfidf = {word: np.log(N / df_tfidf[word]) + 1 for word in vocabulary}

In [81]:
# Compute Document TF-IDF Matrix
tfidf_matrix = np.zeros((N, len(vocabulary)))

for doc_id, doc in enumerate(tokenized_corpus):
    doc_len = len(doc)
    for word in doc:
        tf = doc.count(word) / doc_len
        tfidf_matrix[doc_id, vocab_idx[word]] = tf * idf_tfidf[word]

In [82]:
# Compute Query Vector
query_vector = np.zeros(len(vocabulary))

for word in tokenized_query:
    if word in vocab_idx:
        tf = tokenized_query.count(word) / len(tokenized_query)
        query_vector[vocab_idx[word]] = tf * idf_tfidf[word]

tfidf_scores = [cosine_similarity(query_vector, doc_vec) for doc_vec in tfidf_matrix]
tfidf_ranks = np.argsort(tfidf_scores)[::-1][:2]

Task A: Direct Output

In [83]:
print("TF-IDF Search Results")
for i, idx in enumerate(tfidf_ranks):
    print(f"Rank {i+1}: Doc ID {idx} (Score: {tfidf_scores[idx]:.4f}) - \"{corpus[idx]}\"")

TF-IDF Search Results
Rank 1: Doc ID 1 (Score: 0.4243) - "Python is an interpreted, high-level programming language for general-purpose programming."
Rank 2: Doc ID 3 (Score: 0.2896) - "Information retrieval systems evaluate term frequency and inverse document frequency."


### Task B: BM25 Lexical Ranking
Optimize lexical weights using the Okapi BM25 schema. We penalize excessive term redundancy with saturation constant \
$k_1 = 1.5$ and adjust for length variances using document scaling hyperparameter $b = 0.75$.

In [84]:
k1 = 1.5
b = 0.75
bm25_scores = np.zeros(N)

for doc_id, doc in enumerate(tokenized_corpus):
    doc_len = len(doc)
    score = 0.0

    for q_term in tokenized_query:
        if q_term in vocabulary:
            df_q = sum(1 for d in tokenized_corpus if q_term in d)
            idf_q = np.log((N - df_q + 0.5) / (df_q + 0.5) + 1)

            tf_q = doc.count(q_term)
            numerator = tf_q * (k1 + 1)
            denominator = tf_q + k1 * (1 - b + b * (doc_len / avgdl))

            score += idf_q * (numerator / denominator)
    
    bm25_scores[doc_id] = score

bm25_ranks = np.argsort(bm25_scores)[::-1][:2]

Task B: Direct Output

In [85]:
print("BM25 Search Results")
for i, idx in enumerate(bm25_ranks):
    print(f"Rank {i+1}: Doc ID {idx} (Score: {bm25_scores[idx]:.4f}) - \"{corpus[idx]}\"")

BM25 Search Results
Rank 1: Doc ID 1 (Score: 4.2676) - "Python is an interpreted, high-level programming language for general-purpose programming."
Rank 2: Doc ID 3 (Score: 2.8451) - "Information retrieval systems evaluate term frequency and inverse document frequency."


### Task C: Dens Embbedding Retrieval (Ollama Engine)
Execute vector space encoding through a local LLM server API (`nomic-embed-text`) to project strings into continuous, \
dense embeddings that prioritize conceptual overlap over raw keyword intersections.

In [86]:
def get_ollama_embedding(text, model="nomic-embed-text"):
    url = "http://localhost:11434/api/embeddings"

    try:
        response = requests.post(url, json={"model": model, "prompt": text})
        return np.array(response.json()["embedding"])
    
    except Exception as e:
        raise ConnectionError(f"Ensure Ollama is running locally: {e}")

In [87]:
try:
    corpus_embeddings = [get_ollama_embedding(doc) for doc in corpus]
    query_embedding = get_ollama_embedding(query)
    dense_scores = [cosine_similarity(query_embedding, c_emb) for c_emb in corpus_embeddings]
    dense_ranks = np.argsort(dense_scores)[::-1][:2]

except Exception:
    dense_scores = [0.0] * N
    dense_ranks = [0, 1]

Run in terminal

ollama pull nomic-embed-text

Task C: Direct Output

In [88]:
print("Dense Embedding Search Results")
for i, idx in enumerate(dense_ranks):
    print(f"Rank {i+1}: Doc ID {idx} (Score: {dense_scores[idx]:.4f}) - \"{corpus[idx]}\"")

Dense Embedding Search Results
Rank 1: Doc ID 2 (Score: 0.5940) - "The search engine uses a vector database to retrieve relevant documents quickly."
Rank 2: Doc ID 1 (Score: 0.5823) - "Python is an interpreted, high-level programming language for general-purpose programming."


### System Benchmark and Evaluation Output
Print and verify retrieval scores to analyze exactly where keyword matches diverge from deep semantic alignment.

In [89]:
print("TF-IDF Search Results")
for i, idx in enumerate(tfidf_ranks):
    print(f"Rank {i+1}: Doc ID {idx} (Score: {tfidf_scores[idx]:.4f}) - \"{corpus[idx]}\"")

print("\nBM25 Search Results")
for i, idx in enumerate(bm25_ranks):
    print(f"Rank {i+1}: Doc ID {idx} (Score: {bm25_scores[idx]:.4f}) - \"{corpus[idx]}\"")

print("\nDense Embedding Search Results")
for i, idx in enumerate(dense_ranks):
    print(f"Rank {i+1}: Doc ID {idx} (Score: {dense_scores[idx]:.4f}) - \"{corpus[idx]}\"")

TF-IDF Search Results
Rank 1: Doc ID 1 (Score: 0.4243) - "Python is an interpreted, high-level programming language for general-purpose programming."
Rank 2: Doc ID 3 (Score: 0.2896) - "Information retrieval systems evaluate term frequency and inverse document frequency."

BM25 Search Results
Rank 1: Doc ID 1 (Score: 4.2676) - "Python is an interpreted, high-level programming language for general-purpose programming."
Rank 2: Doc ID 3 (Score: 2.8451) - "Information retrieval systems evaluate term frequency and inverse document frequency."

Dense Embedding Search Results
Rank 1: Doc ID 2 (Score: 0.5940) - "The search engine uses a vector database to retrieve relevant documents quickly."
Rank 2: Doc ID 1 (Score: 0.5823) - "Python is an interpreted, high-level programming language for general-purpose programming."
